# Tiny tomography — course edition

[Week 5 student sheet](../seminars/05_basis_and_tomography.md) · [Worked practice solutions](../solutions/practice/05_basis_and_tomography.md) · [Course index](../README.md)

The sheet supplies the 90-minute route and lab rubric. Widget output is optional: edit the displayed parameter calls or use the paper equations. The final noisy-data section is optional and belongs after orthogonal projection in week 11.

# Tiny tomography: what can the measurements tell us?

**Linear algebra seminar · 1-AIN-152/22 · 90 minutes**  
Original pilot prepared for Ján Pastorek, 27 August 2026.

You will reconstruct a tiny image, discover an invisible change to it, and design a new sensor.
The aim is to connect **linear equations, rank, kernel, affine solution sets, and information**.
This is a guided practice notebook, not a secure assessment: later cells reveal checks and explanations.

**Prerequisites:** matrix multiplication, elimination, linear independence, a kernel, and free variables.
Rank–nullity can be consolidated in the debrief. The noise extension is for a later meeting after orthogonality.

**Software:** Python 3, NumPy, Matplotlib. An optional slider uses `ipywidgets` and IPython;
the activity works without it by editing a parameter and rerunning a cell. No account, data download,
paid service, SymPy, or animation renderer is required by this notebook. Use a local Jupyter installation
or another approved notebook environment. No code in this notebook sends data to a service.


## Working agreement and timing

Use the [week 5 seminar route](../seminars/05_basis_and_tomography.md): 0–8 quiz/prediction; 8–18 pair modeling; 18–30 worked start; 30–55 family and kernel investigation; 55–70 sensor design; 70–82 proof and debrief; 82–90 individual transfer. The notebook is an alternative working surface for that same seminar, not additional work.

Write a prediction before the corresponding checking cell. Rotate solver, skeptic and explainer in teams of 2–3. AI and code are allowed in investigations only if the instructor permits them. A printed copy of the equations and student sheet is sufficient.


## 1. The hidden image — predict first

The image is a 2×2 array:

$$\begin{pmatrix}p&q\\r&s\end{pmatrix}.$$

Sensors report the **top row sum 5**, **bottom row sum 9**, **left column sum 6**, and **right column sum 8**.
The entries are **real, nonnegative intensities**; they need not be integers.

1. Do four measurements always determine four unknowns? Explain your prediction in two sentences.
2. What relation must the four measurements satisfy if they describe any image at all?
3. Could two different images produce these measurements? If you think so, try constructing them.

**My prediction and reason:**

_Write here before scrolling to the checking cells._


## From the original column picture to an image

In the [geometric-view notebook](../2_Geometric_view_of_linear_algebra_checkpoint.ipynb), the unknowns were coefficients of matrix columns. The same interpretation works here: each pixel value weights the measurements that would be produced by one bright pixel at that location.

For example, increasing only the top-left pixel changes the top-row and left-column sums, but leaves the other two sums alone. That observation will become the first column of the measurement matrix. Each row, in contrast, describes what one sensor adds up.

Keep both views in mind as you build the model. We are still solving $Ax=b$; the image gives the entries a concrete meaning. Four reported numbers do not automatically mean four independent pieces of information.

## 2. Model the measurements

Use $x=(p,q,r,s)^T$. Write the four scalar equations and build $Ax=b$.
Explain the meaning of one row of $A$ and of its first column.

**Our equations, matrix, and interpretation:**

_Write here. Then compare with the model in the next cell._


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([
    [1, 1, 0, 0],
    [0, 0, 1, 1],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
], dtype=int)
b = np.array([5, 9, 6, 8], dtype=int)
measurement_names = ['Top row', 'Bottom row', 'Left column', 'Right column']

print('A =')
print(A)
print('b =', b)


## 3. Find the whole family, not only one image

1. Use elimination, or solve in terms of $p=t$, to describe **all real solutions**.
2. Find the interval of $t$ for which every pixel is nonnegative.
3. Give two different nonnegative solutions and verify all four measurements.
4. Identify a nonzero vector $v$ with $Av=0$. Explain what adding $v$ does to an image.
5. Explain the difference between the dimension of the kernel and the number of vectors in it.

**Our family and justification:**

_Write here. Use the checker only after deriving candidates._


In [ ]:
def check_candidate(x):
    """Numerical check of a proposed image; not a proof of completeness."""
    x = np.asarray(x, dtype=float)
    if x.shape != (4,) or not np.isfinite(x).all():
        raise ValueError('Supply four finite numbers in the order p, q, r, s.')
    measured = A @ x
    residual = measured - b
    return {
        'pixels': x.tolist(),
        'measurements': measured.tolist(),
        'max_absolute_residual': float(np.max(np.abs(residual))),
        'matches_to_tolerance': bool(np.allclose(measured, b, atol=1e-10, rtol=0)),
        'nonnegative': bool(np.all(x >= 0)),
    }

candidate = None  # Replace with your own [p, q, r, s], then rerun.
if candidate is None:
    print('Enter a candidate after deriving it, e.g. candidate = [p, q, r, s].')
else:
    print(check_candidate(candidate))


## 4. Reveal and explore — only after your derivation

A solution family is

$$x(t)=(t,5-t,6-t,3+t)^T=x(0)+t(1,-1,-1,1)^T.$$

For nonnegative pixels, $0\leq t\leq 5$.

**Before running the visual:** predict which pixels get brighter when $t$ increases and which sensor
readings change. Then choose two visibly different images with exactly the same readings.

The display uses a fixed grayscale and numerical pixel labels. A change of contrast scale must not
create a false impression that measurements changed. Darker pixels have larger values here.


### Reading the family one equation at a time

After trying the elimination, compare it with this route. Let $p=t$ be the value we have not yet determined. The top-row sum gives $q=5-t$. The left-column sum gives $r=6-t$. Then the bottom-row sum gives $s=9-r=3+t$.

Substitute these into the remaining equation: $q+s=(5-t)+(3+t)=8$. It is satisfied for **every** $t$, so it does not determine the missing value. This is why four equations have not pinned down four unknowns: one of the reported constraints repeats information supplied by the others.

The inequalities now have a different role. They express physical assumptions about brightness, not additional equalities: $t\ge0$, $5-t\ge0$, $6-t\ge0$, $3+t\ge0$. Together they leave $0\le t\le5$. The whole real solution set is a line; the nonnegative images form a segment of that line.

In [ ]:
def solution_family(t):
    t = float(t)
    if not np.isfinite(t):
        raise ValueError('t must be finite.')
    return np.array([t, 5-t, 6-t, 3+t], dtype=float)

def draw_image(t=2.0):
    t = float(t)
    if not 0 <= t <= 5:
        raise ValueError('Use 0 <= t <= 5 for nonnegative pixels.')
    x = solution_family(t)
    measured = A @ x
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), layout='constrained')
    ax = axes[0]
    ax.imshow(x.reshape(2, 2), cmap='Greys', vmin=0, vmax=8)
    for row in range(2):
        for col in range(2):
            value = x.reshape(2, 2)[row, col]
            label = ('p', 'q', 'r', 's')[2*row + col]
            ax.text(col, row, f'{label} = {value:.2f}', ha='center', va='center',
                    color='white' if value > 4 else 'black', fontsize=14)
    ax.set_xticks([0, 1], ['Left', 'Right'])
    ax.set_yticks([0, 1], ['Top', 'Bottom'])
    ax.set_title(f'Different possible images: t = {t:.2f}')
    ax.set_xticks([-.5, .5, 1.5], minor=True)
    ax.set_yticks([-.5, .5, 1.5], minor=True)
    ax.grid(which='minor', color='#777777', linewidth=1)
    ax.tick_params(which='minor', length=0)

    ax = axes[1]
    bars = ax.barh(measurement_names, measured, color='#35666a')
    ax.bar_label(bars, labels=[f'{z:.2f}' for z in measured], padding=5)
    ax.set_xlim(0, 10.5)
    ax.invert_yaxis()
    ax.set_xlabel('Measured sum')
    ax.set_title('Exactly the same four measurements')
    ax.spines[['top', 'right']].set_visible(False)
    return fig

t = 2.0  # Change t between 0 and 5, then rerun if the slider is unavailable.
fig = draw_image(t)
plt.show()
plt.close(fig)


In [ ]:
# Optional slider. The preceding cell is the complete no-widget alternative.
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print('Optional widgets are unavailable. Change t in the preceding cell and rerun it.')
else:
    def explore(t):
        figure = draw_image(t)
        plt.show()
        plt.close(figure)

    slider = widgets.FloatSlider(value=2.0, min=0.0, max=5.0, step=0.1,
                                 description='t', continuous_update=False)
    display(widgets.interactive(explore, t=slider))


### Explain what the experiment cannot prove

Testing several values does not prove that the displayed family contains **all** solutions.
Use your equations or row reduction for that claim.

1. Show that row 1 + row 2 = row 3 + row 4. Why does this imply rank at most 3?
2. Show that the first three rows are independent. Conclude rank exactly 3.
3. Connect this to $\dim\ker A=1$. Does that mean the kernel has one vector?
4. Is the set of all real solutions a vector space? Is the nonnegative solution set a vector space?

**Our explanation:**

_Write here._


## 5. Buy one extra sensor

Your team may buy exactly one sensor. Consider these choices:

| Sensor | New quantity measured |
|---|---|
| Total brightness | $p+q+r+s$ |
| Main diagonal | $p+s$ |
| One pixel | $p$ |

1. Which sensor choices can remove the ambiguity? Justify your answer before calculating anything.
2. For a general sensor $w^Tx$, find a condition involving the kernel direction $v$ that tells you
   whether it supplies the missing information.
3. The main-diagonal sensor reports $p+s=7$. Recover the image and check all five measurements.
4. A colleague says “Every extra measurement makes the reconstruction unique.” Supply a counterexample.

**Our choice, criterion, reconstruction, and verification:**

_Write here. The next cell checks the criterion after you derive it._


In [ ]:
null_direction = np.array([1, -1, -1, 1], dtype=int)
sensor_vectors = {
    'Total brightness': np.array([1, 1, 1, 1], dtype=int),
    'Main diagonal': np.array([1, 0, 0, 1], dtype=int),
    'Top-left pixel': np.array([1, 0, 0, 0], dtype=int),
}
for name, w in sensor_vectors.items():
    print(f'{name}: change in reading when adding v = {w @ null_direction}')

# Optional check of your reconstruction. Supply your own candidate above.
if candidate is not None:
    print('Your main-diagonal sum:', sensor_vectors['Main diagonal'] @ np.asarray(candidate))


## 6. Generalize and challenge

Suppose a consistent system has all real solutions $x=x_0+tv$ with $v\neq0$ and $t\in\mathbb R$.
An extra sensor reports $w^Tx=c$.

Explain all three possibilities: exactly one solution, the same family of solutions, or no solution.
State the relevant conditions; do not assume the new reading is consistent. For nonnegative images,
also check that a recovered parameter lies in the feasible interval.

**Our general statement:**

_Write here._


## 7. Individual transfer response — 8 minutes

Keep the original four measurements, but replace the extra sensor with **$2p+q=9$**.

Without AI, code, or a symbolic solver:

1. Decide whether the new sensor removes the ambiguity. Explain using the kernel direction.
2. Find the image, or prove that none exists.
3. Verify the new reading and one of the original readings.
4. Explain why simply repeating the original top-row measurement would not achieve the same thing.

**My response:**

_Write here. Your instructor will collect individual reasoning, not the completed notebook output._

Suggested four-point rubric: relevant condition; correct conclusion; valid verification; interpretation
of informative versus redundant measurement. Equivalent arguments are welcome. An accessible written
or oral equivalent can be agreed in advance; mathematical reasoning is the criterion, not presentation style.


## Optional later revisit: noisy measurements

**Not part of the 90-minute core. Allow 15–20 minutes after orthogonality/least squares has been introduced.**

Change only the last original sensor reading from 8 to 8.2. Let the changed vector be $b'$.

1. Without solving, prove that $Ax=b'$ has no exact solution.
2. Try the vector $y=(1,1,-1,-1)^T$. Compute $y^TA$ and $y^Tb'$ and explain the contradiction.
3. What might “best approximation” mean? Here we use the **unweighted squared Euclidean residual**.
   That assumes the four readings are treated equally; other noise models may justify different weights.
4. Does a numerical least-squares routine now return a unique mathematical solution? Separate
   an exact solution, a least-squares minimizer, and a minimum-norm least-squares minimizer.

**My prediction:**

_Write here before running the next cell._


In [ ]:
b_noisy = b.astype(float)
b_noisy[-1] = 8.2
y = np.array([1, 1, -1, -1], dtype=int)
print('y^T A =', y @ A)
print('y^T b_noisy =', y @ b_noisy)

# Numerical verification. This routine chooses a minimum-norm minimizer.
x_minimum_norm, _, numerical_rank, singular_values = np.linalg.lstsq(A, b_noisy, rcond=None)
residual = b_noisy - A @ x_minimum_norm
alternative = x_minimum_norm + null_direction
print('One minimum-norm least-squares minimizer:', x_minimum_norm)
print('Residual b_noisy - A x:', residual)
print('Numerical rank:', numerical_rank)
print('A second minimizer has the same fitted measurements:',
      np.allclose(A @ alternative, A @ x_minimum_norm, atol=1e-10, rtol=0))
print('Residual squared norm:', float(residual @ residual))


### Debrief

The unweighted residual is $-0.05(1,1,-1,-1)^T$; its squared norm is 0.01.
All least-squares minimizers have the same fitted measurements, but adding any kernel vector gives
another minimizer. The minimum-norm criterion selects one of them; it does not create new sensor information.
Numerical checks support the reasoning; $y^TA=0$ and $y^Tb'\neq0$ are the exact inconsistency argument.

**One thing I understand differently now:** _Write here._

**One question I still have:** _Write here._

## Context and sources

The numerical instance, questions, and code here were created for this pilot. It is a deliberately tiny
inverse problem, not a realistic medical imaging model. Physical tomography requires additional modeling.

- [Stanford ENGR108](https://web.stanford.edu/class/engr108/) uses matrix methods in applications including tomography.
- [Georgia Tech: solution sets](https://textbooks.math.gatech.edu/ila/solution-sets.html) connects systems to their geometry.
- [NumPy least-squares documentation](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html) describes minimum-norm selection.

The wider design and instructor notes are in `Linear_Algebra_Seminar_Redesign_2026.md`.
